In [5]:
import cv2
import numpy as np

# Load image
img = cv2.imread("test today nb.png")  # BGR format in OpenCV

# Split channels
b, g, r = cv2.split(img)

# Define threshold for red detection
threshold = 100  # tweak as needed

# Create binary mask: 1 if red is strong and green/blue are weak
binary_map = np.where((r > threshold) & (g < threshold) & (b < threshold), 1, 0)

# Optional: save as image
cv2.imwrite("test today_binary_map.png", (binary_map * 255).astype(np.uint8))


True

In [6]:
print(np.shape(img))

(992, 1056, 3)


In [20]:
import rasterio
from rasterio.features import shapes
import numpy as np
from skimage import measure, morphology
from shapely.geometry import shape, mapping
import fiona
import json
import geopandas as gpd

# --------------------------
# Parameters
# --------------------------
input_raster = "test today_binary_map.png"  # binary or probability raster
output_geojson = "test today_binary_map.geojson"
threshold = 0.5                     # if probability raster
simplify_tolerance = 2.0            # units in arbitrary coordinate system
min_area_pixels = 20                 # remove small noise

# --------------------------
# Load raster
# --------------------------
with rasterio.open(input_raster) as src:
    raster = src.read(1)

# --------------------------
# Arbitrary transform
# --------------------------
# Let's assume top-left corner at (1000, 1000), each pixel = 1 unit
from rasterio.transform import from_origin
transform = from_origin(1000, 1000, 1, 1)

# --------------------------
# Threshold if raster is soft probability
# --------------------------
mask = raster > threshold

# --------------------------
# Remove small objects (noise)
# --------------------------
mask = morphology.remove_small_objects(mask.astype(bool), min_size=min_area_pixels)

# --------------------------
# Label connected components
# --------------------------
labels = measure.label(mask)

# --------------------------
# Polygonize each connected component
# --------------------------
polygons = []
for region in measure.regionprops(labels):
    coords = region.coords
    single_mask = np.zeros_like(mask, dtype=np.uint8)
    single_mask[tuple(coords.T)] = 1
    for geom, val in shapes(single_mask, mask=single_mask, transform=transform):
        if val == 1:
            poly = shape(geom)
            poly = poly.simplify(simplify_tolerance, preserve_topology=True)
            polygons.append(poly)

# --------------------------
# Remove empty geometries
# --------------------------
polygons = [poly for poly in polygons if poly.area > 0]

1.475249,1.457421, 103.836006, 103.816916
# Example bounding box (lat/lon)
min_lat, max_lat =  1.475249, 1.457421,
min_lon, max_lon = 103.816916, 103.836006

# min_lat, max_lat = 1.475249, 1.457421
# min_lon, max_lon = 103.836006, 103.816916

# min_lat, max_lat = 1.475249, 1.457421
# min_lon, max_lon = 103.816916, 103.836006

# Determine pixel bounds from mask size
height, width = mask.shape

# Affine mapping function
from shapely.ops import transform as shp_transform

def pixel_to_geo(x, y, z=None):
    lon = min_lon + (x / width) * (max_lon - min_lon)
    lat = max_lat - (y / height) * (max_lat - min_lat)  # flip y
    return lon, lat

# Apply after polygonization
polygons = [shp_transform(pixel_to_geo, poly) for poly in polygons]
gdf = gpd.GeoDataFrame(geometry=polygons)
gdf = gdf.set_crs("EPSG:4326")  # now CRS matches coordinates
gdf.to_file("buildings_real.geojson", driver="GeoJSON")


# --------------------------
# Save to GeoJSON
# --------------------------
# geojson_features = []
# for idx, poly in enumerate(polygons):
#     geojson_features.append({
#         "type": "Feature",
#         "geometry": mapping(poly),
#         "properties": {"id": idx}
#     })

# geojson_dict = {
#     "type": "FeatureCollection",
#     "features": geojson_features
# }

# with open(output_geojson, "w") as f:
#     json.dump(geojson_dict, f)

# print(f"Polygonized buildings saved to {output_geojson}")


c:\Users\tanle\Documents\GitHub\Segmentation\.venv\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


In [8]:
import json
from shapely.geometry import shape, mapping
from shapely.ops import transform
import pyproj

# --------------------------
# Load existing GeoJSON
# --------------------------
input_geojson = "test today_binary_map.geojson"
output_geojson_4326 = "test today_binary_map4326.geojson"

with open(input_geojson) as f:
    data = json.load(f)

# --------------------------
# Define projections
# --------------------------
# Treat the current arbitrary coordinates as EPSG:3857 (meters)
src_crs = pyproj.CRS("EPSG:3857")
dst_crs = pyproj.CRS("EPSG:4326")  # WGS84 lat/lon
project = pyproj.Transformer.from_crs(src_crs, dst_crs, always_xy=True).transform

# --------------------------
# Transform all polygons
# --------------------------
features_4326 = []
for feat in data["features"]:
    geom = shape(feat["geometry"])
    geom_4326 = transform(project, geom)
    features_4326.append({
        "type": "Feature",
        "geometry": mapping(geom_4326),
        "properties": feat["properties"]
    })

geojson_4326 = {
    "type": "FeatureCollection",
    "features": features_4326
}

# --------------------------
# Save transformed GeoJSON
# --------------------------
with open(output_geojson_4326, "w") as f:
    json.dump(geojson_4326, f)

print(f"Polygons converted to EPSG:4326 and saved to {output_geojson_4326}")


Polygons converted to EPSG:4326 and saved to test today_binary_map4326.geojson


In [11]:
print(img.shape)
print(img.shape[0], img.shape[1])

(992, 1056, 3)
992 1056


In [10]:
from shapely.ops import transform
import json

# load geojson with pixel coords
with open("test today_binary_map4326.geojson") as f:
    data = json.load(f)

# local bounds (pixel space)
width, height = img.shape[0], img.shape[1]
1.475249,1.457421, 103.836006, 103.816916
min_lon, min_lat = 103.816916, 1.457421
max_lon, max_lat = 103.836006, 1.475249

def local_to_lonlat(x, y, z=None):
    lon = min_lon + (x / width) * (max_lon - min_lon)
    lat = max_lat - (y / height) * (max_lat - min_lat)  # flip y
    return lon, lat

features_4326 = []
for feat in data["features"]:
    geom = shape(feat["geometry"])
    geom_4326 = transform(local_to_lonlat, geom)   # map pixel coords -> lat/lon
    features_4326.append({
        "type": "Feature",
        "geometry": mapping(geom_4326),
        "properties": feat.get("properties", {})
    })

# save as geojson (already in EPSG:4326)
with open("rectified.geojson", "w") as f:
    json.dump({"type":"FeatureCollection", "features":features_4326}, f)


In [12]:
import geopandas as gpd
from shapely.ops import transform

# Load your local GeoJSON (pixel coordinates)
gdf = gpd.read_file("test today_binary_map4326.geojson")

# Define the bounds in pixel space (adjust if needed)
width, height = img.shape[0], img.shape[1]  # replace with your raster size

# Define your target lat/lon bounds
min_lat, max_lat = 1.457421, 1.475249
min_lon, max_lon = 103.816916, 103.836006

# Define transformation function
def pixel_to_geo(x, y, z=None):
    lon = min_lon + (x / width) * (max_lon - min_lon)
    lat = max_lat - (y / height) * (max_lat - min_lat)  # flip y-axis
    return lon, lat

# Apply to every geometry
gdf["geometry"] = gdf["geometry"].apply(lambda geom: transform(pixel_to_geo, geom))

# Assign CRS to EPSG:4326
gdf = gdf.set_crs("EPSG:4326")

# Save result
gdf.to_file("mapped_back.geojson", driver="GeoJSON")


In [22]:
import rasterio
from rasterio.features import shapes
import numpy as np
from skimage import measure, morphology
from shapely.geometry import shape
from shapely.ops import transform as shp_transform
import geopandas as gpd

# --------------------------
# Parameters
# --------------------------
input_raster = "test_today_binary_map.png"  # binary or probability raster
output_geojson = "test_today_binary_map.geojson"
threshold = 0.5                     # if probability raster
simplify_tolerance = 2.0            # units in pixel space
min_area_pixels = 20                 # remove small noise

# --------------------------
# Real-world bounding box (lat/lon)
# --------------------------
min_lat, max_lat = 1.457421, 1.475249
min_lon, max_lon = 103.816916, 103.836006

# --------------------------
# Load raster
# --------------------------
with rasterio.open(input_raster) as src:
    raster = src.read(1)

# --------------------------
# Threshold / binary mask
# --------------------------
mask = raster > threshold

# --------------------------
# Remove small objects
# --------------------------
mask = morphology.remove_small_objects(mask.astype(bool), min_size=min_area_pixels)

# --------------------------
# Label connected components
# --------------------------
labels = measure.label(mask)

# --------------------------
# Image dimensions
# --------------------------
height, width = mask.shape

# --------------------------
# Pixel -> Geo transform
# --------------------------
def pixel_to_geo(x, y, z=None):
    lon = min_lon + (x / (width - 1)) * (max_lon - min_lon)
    lat = max_lat - (y / (height - 1)) * (max_lat - min_lat)  # flip Y
    return lon, lat

# --------------------------
# Polygonize each component
# --------------------------
polygons = []
for region in measure.regionprops(labels):
    coords = region.coords
    single_mask = np.zeros_like(mask, dtype=np.uint8)
    single_mask[tuple(coords.T)] = 1
    for geom, val in shapes(single_mask, mask=single_mask):
        if val == 1:
            poly = shape(geom)
            poly = poly.simplify(simplify_tolerance, preserve_topology=True)
            poly = shp_transform(pixel_to_geo, poly)  # map to real lat/lon
            polygons.append(poly)

# --------------------------
# Remove empty geometries
# --------------------------
polygons = [poly for poly in polygons if poly.area > 0]

# --------------------------
# Save to GeoJSON via GeoPandas
# --------------------------
gdf = gpd.GeoDataFrame(geometry=polygons)
gdf = gdf.set_crs("EPSG:4326")
gdf.to_file(output_geojson, driver="GeoJSON")

print(f"Polygonized buildings saved to {output_geojson}")


c:\Users\tanle\Documents\GitHub\Segmentation\.venv\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Polygonized buildings saved to test_today_binary_map.geojson
